[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S08/2_Practica/en_clase/S08_02_random_forest_en_clase.ipynb)

# S08 · 02 — Random Forest: la sabiduría de muchos árboles · versión en clase

**Tiempo estimado:** 30 min en clase · **Sprint 8** · M3 (UD 3.7)

📖 *¿El código se hace largo? Léelo [explicado por escrito](https://github.com/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S08/2_Practica/codigo_explicado/S08_02_random_forest.md): recorrido, glosario y qué hace cada celda.* <!--codigo-explicado-->

**Contexto TurisData.** El árbol podado nos dio un AUC de ≈ 0,72 y era muy inestable. Si en lugar de
**un** árbol preguntamos a **cientos** de árboles distintos y hacemos votar, los errores
individuales se compensan. Eso es un **bosque aleatorio** (*Random Forest*).

**Al terminar sabrás:**
1. Explicar el *bagging* (muestras con reemplazo) y la aleatoriedad en las variables.
2. Ajustar `n_estimators`, `max_depth` y `min_samples_leaf`, y usar la puntuación **OOB** como validación gratuita.
3. Comparar la importancia de impureza con la **importancia por permutación**.
4. Valorar el coste: rendimiento vs. explicabilidad y tiempo.

> 🏫 **Versión «en clase»** · ⏱ ≈ 30 min <!--reparto-->
>
> - Es **la parte de este cuaderno que se hace en el aula**, con el docente: ejercicios **1, 2**.
> - Lo demás (ejercicios 🏠 y mini-reto) está en el [cuaderno completo](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S08/2_Practica/para_casa/S08_02_random_forest.ipynb), para hacer en casa.


In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore", category=UserWarning)
SEMILLA = 42
sns.set_theme(style="whitegrid")

df = pd.read_csv(dato("reservas_turisdata.csv"), parse_dates=["fecha_reserva", "fecha_llegada"])
df["temporada_alta"] = df["fecha_llegada"].dt.month.isin([1, 2, 3, 7, 8, 12]).astype(int)
columnas_num = ["antelacion_dias", "noches", "adultos", "ninos", "precio_noche",
                "peticiones_especiales", "cliente_repetidor", "cancelaciones_previas", "temporada_alta"]
columnas_cat = ["tipo_habitacion", "regimen", "canal", "pais_origen", "tarifa", "deposito"]
X, y = df[columnas_num + columnas_cat], df["cancelada"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEMILLA, stratify=y)
cv = StratifiedKFold(5, shuffle=True, random_state=SEMILLA)

prep = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), columnas_num),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), columnas_cat),
])
Xtr = prep.fit_transform(X_train)
Xte = prep.transform(X_test)
nombres = prep.get_feature_names_out()

## 1. La idea: bagging + variables al azar

Un bosque construye muchos árboles y **promedia** sus predicciones. Para que los árboles sean
*distintos entre sí* (si fueran iguales, promediar no serviría), se introduce aleatoriedad de dos formas:
1. **Bagging** (*bootstrap aggregating*): cada árbol se entrena con una **muestra con reemplazo** de las reservas
   (unas aparecen repetidas y otras, ≈ 37 %, no aparecen).
2. En cada pregunta, cada árbol solo puede elegir entre un **subconjunto aleatorio de variables** (`max_features`).

Cada árbol individual es peor que uno bien podado, pero **sus errores no coinciden**; al votar, el ruido se
cancela y baja la varianza. Las reservas que un árbol no vio (**out-of-bag**, OOB) sirven para evaluar
sin necesidad de validación cruzada.

In [ ]:
# Cuántas reservas quedan fuera de una muestra bootstrap: ≈ 36,8 %
rng = np.random.default_rng(SEMILLA)
n = 5000
fuera = len(set(range(n)) - set(rng.integers(0, n, n))) / n
print(f"Reservas que NO entran en una muestra bootstrap: {fuera:.1%}")

**Ejercicio 1 · 🏫 en clase.** Entrena `bosque = RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
random_state=SEMILLA, n_jobs=2)` con train. Guarda `auc_test_bosque` (AUC en test con `predict_proba`) y
el AUC en test del árbol podado de la práctica anterior (`DecisionTreeClassifier(min_samples_leaf=80,
random_state=SEMILLA)`) en `auc_test_arbol`.

In [ ]:
# TODO: fit sobre Xtr, y_train; roc_auc_score(y_test, modelo.predict_proba(Xte)[:, 1]) para ambos
...
assert auc_test_bosque > auc_test_arbol, "El bosque debería superar a un único árbol podado"
assert auc_test_bosque > 0.70, "El AUC del bosque debería superar 0,70"
print(f"Árbol podado: {auc_test_arbol:.3f} · Bosque de 300 árboles: {auc_test_bosque:.3f}")

## 2. Cuántos árboles y de qué tamaño

- `n_estimators`: **más árboles casi nunca perjudica** (solo cuesta tiempo); se estabiliza.
- `min_samples_leaf` y `max_depth`: controlan el sobreajuste de cada árbol.
- `max_features`: nº de variables por pregunta (por defecto `sqrt` del total en clasificación).

Usamos la puntuación **OOB** (`oob_score=True`) para ver la evolución sin validación cruzada.

In [ ]:
n_arboles = [5, 10, 25, 50, 100, 200, 400]
oob_auc = []
for k in n_arboles:
    b = RandomForestClassifier(n_estimators=k, min_samples_leaf=5, oob_score=True,
                               random_state=SEMILLA, n_jobs=2).fit(Xtr, y_train)
    # oob_decision_function_ contiene la probabilidad OOB de cada reserva de train; con pocos árboles, algunas
    # reservas no quedan fuera en ninguno (NaN) y se rellenan con 0,5 («no sé»)
    oob_auc.append(roc_auc_score(y_train, np.nan_to_num(b.oob_decision_function_[:, 1], nan=0.5)))
plt.figure(figsize=(7, 3.8))
plt.plot(n_arboles, oob_auc, marker="o", color="#2a6f97")
plt.xscale("log")
plt.title("Random Forest: AUC out-of-bag según el número de árboles")
plt.xlabel("Número de árboles (escala logarítmica)")
plt.ylabel("AUC out-of-bag")
plt.show()

**Ejercicio 2 · 🏫 en clase.** Con la lista `oob_auc`, guarda en `mejora_5_a_400` cuánto sube el AUC OOB al pasar de 5 a
400 árboles (`oob_auc[-1] - oob_auc[0]`) y en `mejora_100_a_400` lo que sube de 100 a 400 árboles.
Fíjate en que la segunda es mucho menor: los rendimientos son decrecientes.

In [ ]:
# TODO: los índices de n_arboles: 5 está en la posición 0, 100 en la 4 y 400 en la última
...
assert mejora_5_a_400 > 0.05, "De 5 a 400 árboles el AUC debería subir bastante"
assert abs(mejora_100_a_400) < mejora_5_a_400 / 2, "De 100 a 400 la mejora es mucho menor (se satura)"
print(f"5 → 400 árboles: +{mejora_5_a_400:.3f} · 100 → 400 árboles: {mejora_100_a_400:+.3f}")

---

## 🏁 Fin de la parte de clase

- Lo que queda (los ejercicios 🏠 y el mini-reto) está en el **[cuaderno completo](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S08/2_Practica/para_casa/S08_02_random_forest.ipynb)**.
- Ahí también tienes, en cada ejercicio de casa, su celda plegada «🆘 Solución de referencia» por si te quedas atascado.
